# 02 SAJU ML Feature Parity Audit V4

**Notebook version:** `SAJU_ML_FEATURE_PARITY_AUDIT_V4_20260815`

## Purpose

Before building a V4 model, verify that the feature representation itself has been stable.

The core question is not:

> "Are the feature values approximately similar?"

It is:

> **For the same person and event year, does the current canonical engine reproduce the exact frozen Ten-God 10-vector used by the historical ML caches?**

## Strict canonical parity scope

Strict re-extraction is required for:

- V2 `NEW_DEV` only
- consumed `NEW_DEV_2`

These pools contain exact civil birth time, UTC offset, longitude, and event-year feature caches.

## V1 legacy treatment

V1 is handled separately.

Some V1 subjects do not contain enough canonical geospatial/offset provenance to guarantee exact reconstruction of
the historical solar-time pipeline. Therefore this notebook does **not** pretend V1 has full modern canonical parity.

For V1 it checks:

- frozen Ten-God columns exist
- values are binary
- Daewoon group is one-hot
- Sewoon group is one-hot
- `(subject_id, year)` keys are unique

The final decision explicitly records V1 as `LEGACY_CANONICAL_PARITY_NOT_CLAIMED` unless a future verified V1 engine-input
package is created.

## Hard rule

For V2 NEW_DEV and NEW_DEV_2:

**1 mismatch in the frozen 10 features = strict parity FAIL.**

No tolerance, no "mostly matching".

In [1]:
from pathlib import Path
from datetime import datetime
import hashlib
import json
import sys
import warnings

import numpy as np
import pandas as pd

from IPython.display import display

warnings.filterwarnings("ignore")

NOTEBOOK_VERSION = "SAJU_ML_FEATURE_PARITY_AUDIT_V4_20260815"

TENGOD_COLS = [
    "tengod__dw_group_peer",
    "tengod__dw_group_output",
    "tengod__dw_group_wealth",
    "tengod__dw_group_officer",
    "tengod__dw_group_resource",
    "tengod__sw_group_peer",
    "tengod__sw_group_output",
    "tengod__sw_group_wealth",
    "tengod__sw_group_officer",
    "tengod__sw_group_resource",
]

GROUPS = ["peer", "output", "wealth", "officer", "resource"]

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p] + list(p.parents):
        if (candidate / "saju_engine.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find repo root containing saju_engine.py. "
        "Run this notebook from inside the Chartpalja saju repository."
    )

REPO_ROOT = find_repo_root()

ENGINE_PATH = REPO_ROOT / "saju_engine.py"

V1_BASE_DIR = REPO_ROOT / "research/ml/artifacts/v1"

V2_DEV_CORPUS_PATH = (
    REPO_ROOT / "research/ml_corpus/v2/SAJU_ML_CORPUS_V2_NEW_DEV.json"
)
V2_FROZEN_CACHE_PATH = (
    REPO_ROOT / "research/ml/artifacts/v2/V2_NEW_DEV_feature_table_R2.csv"
)
V2_ALL_FEATURE_CACHE_PATH = (
    REPO_ROOT
    / "research/ml/artifacts/v3_all_feature_tournament/V2_NEW_DEV_all_feature_table_v3.csv"
)

NEW2_CORPUS_PATH = (
    REPO_ROOT / "research/ml_corpus/new_dev_2/SAJU_ML_NEW_DEV_2_CORPUS.json"
)
NEW2_CACHE_PATH = (
    REPO_ROOT
    / "research/ml/artifacts/v3_new_dev_2_replication/NEW_DEV_2_feature_table_v3.csv"
)

OUT = REPO_ROOT / "research/ml/artifacts/v4_feature_parity_audit"
OUT.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("=" * 88)
print("NOTEBOOK VERSION:", NOTEBOOK_VERSION)
print("repo:", REPO_ROOT)
print("engine sha256:", sha256_file(ENGINE_PATH))
print("output:", OUT)
print("=" * 88)

NOTEBOOK VERSION: SAJU_ML_FEATURE_PARITY_AUDIT_V4_20260815
repo: /Users/sangjinlee/Desktop/projects/saju
engine sha256: d39e0c4d777ae3a19394c9175319a4f8a6a709c2b4d34400965b618ffbfdad1e
output: /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v4_feature_parity_audit


## 1. Holdout-safe file preflight

The notebook requires a dedicated V2 `NEW_DEV` corpus and never falls back to a combined V2 file.

In [2]:
required = {
    "saju_engine.py": ENGINE_PATH,
    "V2 NEW_DEV-only corpus": V2_DEV_CORPUS_PATH,
    "V2 frozen feature cache R2": V2_FROZEN_CACHE_PATH,
    "NEW_DEV_2 consumed corpus": NEW2_CORPUS_PATH,
    "NEW_DEV_2 feature cache": NEW2_CACHE_PATH,
}

for label, path in required.items():
    ok = path.exists()
    print("%-34s %s  %s" % (label, "OK" if ok else "MISSING", path))
    if not ok:
        raise FileNotFoundError(path)

# V1 cache: one of parquet/csv.
v1_parquet = V1_BASE_DIR / "ml_feature_table_v1.parquet"
v1_csv = V1_BASE_DIR / "ml_feature_table_v1.csv"

if not v1_parquet.exists() and not v1_csv.exists():
    raise FileNotFoundError(
        "Need V1 legacy feature cache: ml_feature_table_v1.parquet or .csv"
    )

# Never configure a sealed input.
for path in [
    V2_DEV_CORPUS_PATH,
    V2_FROZEN_CACHE_PATH,
    NEW2_CORPUS_PATH,
    NEW2_CACHE_PATH,
]:
    upper = str(path).upper()
    assert "NEW_CONFIRM" not in upper
    assert "FINAL" not in upper
    assert "CHECK" not in upper
    assert "VALIDATION_B" not in upper

print("\nHOLDOUT-SAFE PARITY PREFLIGHT: PASS")

saju_engine.py                     OK  /Users/sangjinlee/Desktop/projects/saju/saju_engine.py
V2 NEW_DEV-only corpus             OK  /Users/sangjinlee/Desktop/projects/saju/research/ml_corpus/v2/SAJU_ML_CORPUS_V2_NEW_DEV.json
V2 frozen feature cache R2         OK  /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v2/V2_NEW_DEV_feature_table_R2.csv
NEW_DEV_2 consumed corpus          OK  /Users/sangjinlee/Desktop/projects/saju/research/ml_corpus/new_dev_2/SAJU_ML_NEW_DEV_2_CORPUS.json
NEW_DEV_2 feature cache            OK  /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v3_new_dev_2_replication/NEW_DEV_2_feature_table_v3.csv

HOLDOUT-SAFE PARITY PREFLIGHT: PASS


## 2. V1 legacy cache invariants

This is **not** a canonical re-extraction claim.

In [3]:
if v1_parquet.exists():
    v1_cache = pd.read_parquet(v1_parquet)
    v1_cache_path = v1_parquet
else:
    v1_cache = pd.read_csv(v1_csv)
    v1_cache_path = v1_csv

legacy_checks = []

missing_v1 = [c for c in TENGOD_COLS if c not in v1_cache.columns]

legacy_checks.append({
    "check": "frozen_tengod_columns_present",
    "passed": len(missing_v1) == 0,
    "details": "missing=%s" % missing_v1,
})

if not missing_v1:
    vals = v1_cache[TENGOD_COLS].apply(pd.to_numeric, errors="coerce")

    non_binary = int(
        (~vals.isin([0, 1])).sum().sum()
    )

    dw_sum = vals[
        ["tengod__dw_group_" + g for g in GROUPS]
    ].sum(axis=1)

    sw_sum = vals[
        ["tengod__sw_group_" + g for g in GROUPS]
    ].sum(axis=1)

    legacy_checks.extend([
        {
            "check": "binary_values",
            "passed": non_binary == 0,
            "details": "non_binary_cells=%d" % non_binary,
        },
        {
            "check": "dw_onehot_sum_1",
            "passed": bool((dw_sum == 1).all()),
            "details": "bad_rows=%d" % int((dw_sum != 1).sum()),
        },
        {
            "check": "sw_onehot_sum_1",
            "passed": bool((sw_sum == 1).all()),
            "details": "bad_rows=%d" % int((sw_sum != 1).sum()),
        },
    ])

if {"subject_id", "year"}.issubset(v1_cache.columns):
    duplicate_keys = int(
        v1_cache.duplicated(["subject_id", "year"]).sum()
    )
else:
    duplicate_keys = -1

legacy_checks.append({
    "check": "unique_subject_year_keys",
    "passed": duplicate_keys == 0,
    "details": "duplicate_keys=%d" % duplicate_keys,
})

v1_legacy_checks = pd.DataFrame(legacy_checks)

v1_legacy_checks.to_csv(
    OUT / "V4_V1_LEGACY_CACHE_INVARIANTS.csv",
    index=False,
)

display(v1_legacy_checks)

V1_LEGACY_INVARIANTS_PASS = bool(
    v1_legacy_checks["passed"].all()
)

print(
    "\nV1 status:",
    "LEGACY_INVARIANTS_PASS" if V1_LEGACY_INVARIANTS_PASS
    else "LEGACY_INVARIANTS_FAIL",
)
print("Canonical V1 parity claim: NOT MADE")

,check,passed,details
0,frozen_tengod_columns_present,True,missing=[]
1,binary_values,True,non_binary_cells=0
2,dw_onehot_sum_1,True,bad_rows=0
3,sw_onehot_sum_1,True,bad_rows=0
4,unique_subject_year_keys,True,duplicate_keys=0



V1 status: LEGACY_INVARIANTS_PASS
Canonical V1 parity claim: NOT MADE


## 3. Canonical current-engine extractor

The canonical lineage is explicitly:

`civil birth -> longitude/UTC solar correction -> saju_engine -> annual timeline -> day stem / Daewoon / Sewoon -> grouped Ten-God`

This mirrors the V2/V3 runtime convention and prevents double solar correction.

In [4]:
sys.path.insert(0, str(REPO_ROOT))

import saju_engine as se
import sajupy

STEMS = list("甲乙丙丁戊己庚辛壬癸")
BRANCHES = list("子丑寅卯辰巳午未申酉戌亥")

STEM_ELEMENT = {
    "甲":"wood", "乙":"wood",
    "丙":"fire", "丁":"fire",
    "戊":"earth", "己":"earth",
    "庚":"metal", "辛":"metal",
    "壬":"water", "癸":"water",
}

GENERATES = {
    "wood":"fire",
    "fire":"earth",
    "earth":"metal",
    "metal":"water",
    "water":"wood",
}

CONTROLS = {
    "wood":"earth",
    "earth":"water",
    "water":"fire",
    "fire":"metal",
    "metal":"wood",
}

def parse_utc_offset(raw):
    if isinstance(raw, (int, float)):
        return float(raw)

    text = str(raw).strip()

    sign = -1.0 if text.startswith("-") else 1.0
    text = text[1:] if text[:1] in "+-" else text

    hh, mm = text.split(":")

    return sign * (
        int(hh) + int(mm) / 60.0
    )

def corrected_birth(birth):
    if not birth.get("birth_time_known", False):
        raise ValueError("Exact birth time is required for strict parity.")

    if birth.get("longitude") is None:
        raise ValueError("Longitude is required for strict parity.")

    if birth.get("utc_offset") is None:
        raise ValueError("UTC offset is required for strict parity.")

    y, m, d = map(
        int,
        str(birth["date"]).split("-"),
    )

    hh, minute = map(
        int,
        str(birth["time"]).split(":")[:2],
    )

    longitude = float(
        birth["longitude"]
    )

    utc_hours = parse_utc_offset(
        birth["utc_offset"]
    )

    calculator = sajupy.get_saju_calculator()

    correction = float(
        calculator._calculate_solar_time_correction(
            longitude,
            utc_hours,
        )
    )

    h2, min2, day_change = (
        calculator._adjust_time_for_solar(
            hh,
            minute,
            correction,
        )
    )

    y2, m2, d2 = (
        calculator._adjust_date_for_solar(
            y,
            m,
            d,
            day_change,
        )
    )

    return {
        "year": int(y2),
        "month": int(m2),
        "day": int(d2),
        "hour": int(h2),
        "minute": int(min2),
        "solar_correction_minutes": correction,
    }

def pillar_chars(value):
    if (
        isinstance(value, (list, tuple))
        and len(value) >= 2
    ):
        a, b = str(value[0]), str(value[1])

        if a in STEMS and b in BRANCHES:
            return a, b

    text = str(value)

    stem = next(
        (x for x in text if x in STEMS),
        None,
    )
    branch = next(
        (x for x in text if x in BRANCHES),
        None,
    )

    return stem, branch

def sexagenary_year_pillar(year):
    index = (int(year) - 1984) % 60
    return (
        STEMS[index % 10]
        + BRANCHES[index % 12]
    )

def get_daewoon_pillar(meta_row):
    for key in (
        "대운_pillar",
        "대운",
        "daewoon_pillar",
    ):
        if meta_row.get(key):
            stem, branch = pillar_chars(
                meta_row[key]
            )

            if stem and branch:
                return stem + branch

    return None

def get_sewoon_pillar(meta_row, year):
    for key in (
        "세운_pillar",
        "세운",
        "연주",
        "year_pillar",
    ):
        if meta_row.get(key):
            stem, branch = pillar_chars(
                meta_row[key]
            )

            if stem and branch:
                return stem + branch

    return sexagenary_year_pillar(year)

def ten_god_group(day_stem, other_stem):
    day_element = STEM_ELEMENT[day_stem]
    other_element = STEM_ELEMENT[other_stem]

    if day_element == other_element:
        return "peer"

    if GENERATES[day_element] == other_element:
        return "output"

    if CONTROLS[day_element] == other_element:
        return "wealth"

    if CONTROLS[other_element] == day_element:
        return "officer"

    if GENERATES[other_element] == day_element:
        return "resource"

    raise RuntimeError(
        "Could not map Ten-God group: %s %s"
        % (day_stem, other_stem)
    )

CALCULATOR = sajupy.get_saju_calculator()
ENGINE_MIN_YEAR = int(CALCULATOR.min_year)
ENGINE_MAX_YEAR = int(CALCULATOR.max_year)

def build_engine_subject(subject):
    birth = subject.get("birth") or {}
    corrected = corrected_birth(birth)

    if not (
        ENGINE_MIN_YEAR
        <= corrected["year"]
        <= ENGINE_MAX_YEAR
    ):
        raise ValueError(
            "%s corrected birth year %s outside engine range %s-%s"
            % (
                subject.get("name"),
                corrected["year"],
                ENGINE_MIN_YEAR,
                ENGINE_MAX_YEAR,
            )
        )

    inp = se.BirthInput(
        year=corrected["year"],
        month=corrected["month"],
        day=corrected["day"],
        hour=corrected["hour"],
        minute=corrected["minute"],
        gender=subject["gender"],
        calendar=birth.get("calendar", "solar"),
        is_leap_month=False,
        use_solar_time=False,
        utc_offset=9,
    )

    result = se.compute_all(inp)

    timeline = (
        result["chart_data"]["연도별_타임라인"]
    )

    meta = {
        int(row["year"]): row
        for row in timeline
    }

    return corrected, result, meta

def canonical_row(subject, year, engine_bundle):
    corrected, result, meta = engine_bundle

    year = int(year)

    if year not in meta:
        raise RuntimeError(
            "Annual timeline missing year %s for %s"
            % (year, subject.get("name"))
        )

    meta_row = meta[year]

    day_stem, day_branch = pillar_chars(
        result["원국"]["day"]
    )

    dw_stem, dw_branch = pillar_chars(
        get_daewoon_pillar(
            meta_row
        )
    )

    sw_stem, sw_branch = pillar_chars(
        get_sewoon_pillar(
            meta_row,
            year,
        )
    )

    if not all([
        day_stem,
        day_branch,
        dw_stem,
        dw_branch,
        sw_stem,
        sw_branch,
    ]):
        raise RuntimeError(
            "Pillar parse failure for %s %s"
            % (subject.get("name"), year)
        )

    dw_group = ten_god_group(
        day_stem,
        dw_stem,
    )

    sw_group = ten_god_group(
        day_stem,
        sw_stem,
    )

    out = {
        "canonical_day_stem": day_stem,
        "canonical_day_branch": day_branch,
        "canonical_dw_stem": dw_stem,
        "canonical_dw_branch": dw_branch,
        "canonical_sw_stem": sw_stem,
        "canonical_sw_branch": sw_branch,
        "canonical_dw_group": dw_group,
        "canonical_sw_group": sw_group,
        "corrected_birth_date": "%04d-%02d-%02d" % (
            corrected["year"],
            corrected["month"],
            corrected["day"],
        ),
        "corrected_birth_time": "%02d:%02d" % (
            corrected["hour"],
            corrected["minute"],
        ),
        "solar_correction_minutes": float(
            corrected["solar_correction_minutes"]
        ),
    }

    for group in GROUPS:
        out[
            "canonical__tengod__dw_group_" + group
        ] = float(dw_group == group)

        out[
            "canonical__tengod__sw_group_" + group
        ] = float(sw_group == group)

    return out

## 4. Strict parity runner

The cache is treated as the historical value actually used by ML.
The current engine is treated as the canonical recomputation.

The comparison is exact for the 10 frozen Ten-God values.

In [5]:
def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

v2_corpus = load_json(V2_DEV_CORPUS_PATH)
new2_corpus = load_json(NEW2_CORPUS_PATH)

v2_subject_map = {
    x["subject_id"]: x
    for x in v2_corpus["subjects"]
}

new2_subject_map = {
    x["subject_id"]: x
    for x in new2_corpus["subjects"]
}

v2_cache = pd.read_csv(
    V2_FROZEN_CACHE_PATH
)

new2_cache = pd.read_csv(
    NEW2_CACHE_PATH
)

def validate_cache_structure(
    pool_name,
    cache,
    subject_map,
):
    missing = [
        c for c in (
            ["subject_id", "year"]
            + TENGOD_COLS
        )
        if c not in cache.columns
    ]

    if missing:
        raise RuntimeError(
            "%s cache missing columns: %s"
            % (pool_name, missing)
        )

    duplicates = int(
        cache.duplicated(
            ["subject_id", "year"]
        ).sum()
    )

    if duplicates:
        raise RuntimeError(
            "%s cache contains %d duplicate subject/year rows."
            % (pool_name, duplicates)
        )

    unknown_subject_ids = sorted(
        set(cache["subject_id"])
        - set(subject_map)
    )

    if unknown_subject_ids:
        raise RuntimeError(
            "%s cache references unknown subject IDs: %s"
            % (
                pool_name,
                unknown_subject_ids[:20],
            )
        )

    numeric = cache[
        TENGOD_COLS
    ].apply(
        pd.to_numeric,
        errors="coerce",
    )

    bad_binary = int(
        (~numeric.isin([0, 1]))
        .sum()
        .sum()
    )

    if bad_binary:
        raise RuntimeError(
            "%s cache has %d non-binary TENGOD cells."
            % (pool_name, bad_binary)
        )

    dw_sum = numeric[
        ["tengod__dw_group_" + g for g in GROUPS]
    ].sum(axis=1)

    sw_sum = numeric[
        ["tengod__sw_group_" + g for g in GROUPS]
    ].sum(axis=1)

    if not (dw_sum == 1).all():
        raise RuntimeError(
            "%s cache has invalid DW one-hot rows."
            % pool_name
        )

    if not (sw_sum == 1).all():
        raise RuntimeError(
            "%s cache has invalid SW one-hot rows."
            % pool_name
        )

validate_cache_structure(
    "V2_NEW_DEV",
    v2_cache,
    v2_subject_map,
)

validate_cache_structure(
    "NEW_DEV_2_CONSUMED",
    new2_cache,
    new2_subject_map,
)

print("Cache structural preflight: PASS")

Cache structural preflight: PASS


### Recompute and compare row by row

In [6]:
def run_strict_parity(
    pool_name,
    cache,
    subject_map,
):
    rows = []
    engine_cache = {}

    for i, cache_row in cache.iterrows():
        sid = cache_row["subject_id"]
        year = int(cache_row["year"])
        subject = subject_map[sid]

        if sid not in engine_cache:
            engine_cache[sid] = build_engine_subject(
                subject
            )

        canonical = canonical_row(
            subject,
            year,
            engine_cache[sid],
        )

        row = {
            "pool": pool_name,
            "subject_id": sid,
            "name": subject.get("name"),
            "year": year,
        }

        row.update(canonical)

        mismatch_cols = []

        for col in TENGOD_COLS:
            historical = float(cache_row[col])
            canonical_value = float(
                canonical[
                    "canonical__" + col
                ]
            )

            row[
                "historical__" + col
            ] = historical

            row[
                "match__" + col
            ] = bool(
                historical == canonical_value
            )

            if historical != canonical_value:
                mismatch_cols.append(col)

        row["n_tengod_mismatches"] = len(
            mismatch_cols
        )
        row["mismatch_features"] = "|".join(
            mismatch_cols
        )
        row["strict_tengod_row_match"] = (
            len(mismatch_cols) == 0
        )

        rows.append(row)

        if (i + 1) % 50 == 0:
            print(
                "%s canonical rows %d/%d"
                % (
                    pool_name,
                    i + 1,
                    len(cache),
                )
            )

    return pd.DataFrame(rows)

v2_parity = run_strict_parity(
    "V2_NEW_DEV",
    v2_cache,
    v2_subject_map,
)

new2_parity = run_strict_parity(
    "NEW_DEV_2_CONSUMED",
    new2_cache,
    new2_subject_map,
)

strict_parity = pd.concat(
    [v2_parity, new2_parity],
    ignore_index=True,
)

strict_parity.to_csv(
    OUT / "V4_STRICT_TENGOD_PARITY_ROWS.csv",
    index=False,
)

display(
    strict_parity.groupby("pool").agg(
        rows=("subject_id", "size"),
        subjects=("subject_id", "nunique"),
        exact_rows=("strict_tengod_row_match", "sum"),
        total_feature_mismatches=("n_tengod_mismatches", "sum"),
    ).reset_index()
)

[SAJU_DEBUG] original_input: 1968-03-12 18:41
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1968-03-12
[SAJU_DEBUG] final_datetime(KST): 1968-03-12T18:41:00+09:00
[SAJU_DEBUG] pillars: 연=戊申 월=乙卯 일=辛巳 시=丁酉
[SAJU_DEBUG] original_input: 1961-12-19 15:00
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=female
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1961-12-19
[SAJU_DEBUG] final_datetime(KST): 1961-12-19T15:00:00+09:00
[SAJU_DEBUG] 반시보정: 丙申→乙未 (15:00)
[SAJU_DEBUG] pillars: 연=辛丑 월=庚子 일=丙戌 시=乙未
[SAJU_DEBUG] original_input: 1969-02-11 22:29
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=female
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1969-02-11
[SAJU_DEBUG] final_datetime(KST): 1969-02-11T22:29:00+09:00
[SAJU_DEBUG] pillars: 연=己酉 월=丙寅 일=丁巳 시=辛亥
[SAJU_DEBUG] original_input: 1961-06-01 12:24
[SAJU_DEBUG] calendar=solar, 

V2_NEW_DEV canonical rows 50/216


[SAJU_DEBUG] original_input: 1966-02-20 14:12
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=female
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1966-02-20
[SAJU_DEBUG] final_datetime(KST): 1966-02-20T14:12:00+09:00
[SAJU_DEBUG] pillars: 연=丙午 월=庚寅 일=庚戌 시=癸未
[SAJU_DEBUG] original_input: 1954-01-19 04:30
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=female
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1954-01-19
[SAJU_DEBUG] final_datetime(KST): 1954-01-19T04:30:00+09:00
[SAJU_DEBUG] pillars: 연=癸巳 월=乙丑 일=乙亥 시=戊寅
[SAJU_DEBUG] original_input: 1930-05-31 17:25
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1930-05-31
[SAJU_DEBUG] final_datetime(KST): 1930-05-31T17:25:00+09:00
[SAJU_DEBUG] 반시보정: 丁酉→丙申 (17:25)
[SAJU_DEBUG] pillars: 연=庚午 월=辛巳 일=辛巳 시=丙申
[SAJU_DEBUG] original_input: 1947-03-19 14:17
[SAJU_DEBUG] calendar=solar, 

V2_NEW_DEV canonical rows 100/216


[SAJU_DEBUG] original_input: 1982-04-30 12:03
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=female
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1982-04-30
[SAJU_DEBUG] final_datetime(KST): 1982-04-30T12:03:00+09:00
[SAJU_DEBUG] pillars: 연=壬戌 월=甲辰 일=癸未 시=戊午
[SAJU_DEBUG] original_input: 1967-02-10 07:54
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=female
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1967-02-10
[SAJU_DEBUG] final_datetime(KST): 1967-02-10T07:54:00+09:00
[SAJU_DEBUG] pillars: 연=丁未 월=壬寅 일=乙巳 시=庚辰
[SAJU_DEBUG] original_input: 1974-05-16 16:47
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=female
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1974-05-16
[SAJU_DEBUG] final_datetime(KST): 1974-05-16T16:47:00+09:00
[SAJU_DEBUG] pillars: 연=甲寅 월=己巳 일=丁巳 시=戊申
[SAJU_DEBUG] original_input: 1974-11-11 02:54
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=mal

...

NEW_DEV_2_CONSUMED canonical rows 150/200


[SAJU_DEBUG] original_input: 1924-06-04 06:41
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1924-06-04
[SAJU_DEBUG] final_datetime(KST): 1924-06-04T06:41:00+09:00
[SAJU_DEBUG] pillars: 연=甲子 월=己巳 일=甲寅 시=丁卯
[SAJU_DEBUG] original_input: 1944-12-04 22:02
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1944-12-04
[SAJU_DEBUG] final_datetime(KST): 1944-12-04T22:02:00+09:00
[SAJU_DEBUG] pillars: 연=甲申 월=乙亥 일=壬寅 시=辛亥
[SAJU_DEBUG] original_input: 1942-07-27 07:40
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1942-07-27
[SAJU_DEBUG] final_datetime(KST): 1942-07-27T07:40:00+09:00
[SAJU_DEBUG] pillars: 연=壬午 월=丁未 일=辛巳 시=壬辰
[SAJU_DEBUG] original_input: 1943-07-16 12:56
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJ

NEW_DEV_2_CONSUMED canonical rows 200/200


[SAJU_DEBUG] original_input: 1953-10-09 23:57
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1953-10-09
[SAJU_DEBUG] final_datetime(KST): 1953-10-09T23:57:00+09:00
[SAJU_DEBUG] pillars: 연=癸巳 월=壬戌 일=甲午 시=甲子


,pool,rows,subjects,exact_rows,total_feature_mismatches
0,NEW_DEV_2_CONSUMED,200,100,200,0
1,V2_NEW_DEV,216,70,216,0


## 5. Optional dynamic pillar one-hot parity

If the V2 all-feature V3 cache exists, verify its raw dynamic pillar one-hots too.

This is useful because a Ten-God mismatch can originate from either:

- the pillar itself changing, or
- the Ten-God mapping changing.

In [7]:
pillar_parity_rows = []

if V2_ALL_FEATURE_CACHE_PATH.exists():
    v2_all = pd.read_csv(
        V2_ALL_FEATURE_CACHE_PATH
    )

    canonical_lookup = v2_parity.set_index(
        ["subject_id", "year"]
    )

    vocab = {
        "dw_stem": STEMS,
        "dw_branch": BRANCHES,
        "sw_stem": STEMS,
        "sw_branch": BRANCHES,
    }

    for _, r in v2_all.iterrows():
        key = (
            r["subject_id"],
            int(r["year"]),
        )

        if key not in canonical_lookup.index:
            continue

        c = canonical_lookup.loc[key]

        if isinstance(c, pd.DataFrame):
            raise RuntimeError(
                "Unexpected duplicate canonical parity key: %s"
                % (key,)
            )

        row = {
            "subject_id": key[0],
            "year": key[1],
        }

        mismatch = []

        canonical_values = {
            "dw_stem": c["canonical_dw_stem"],
            "dw_branch": c["canonical_dw_branch"],
            "sw_stem": c["canonical_sw_stem"],
            "sw_branch": c["canonical_sw_branch"],
        }

        for family, values in vocab.items():
            prefix = "basic__" + family + "_"

            for value in values:
                col = prefix + value

                if col not in v2_all.columns:
                    continue

                expected = float(
                    canonical_values[family]
                    == value
                )

                observed = float(
                    r[col]
                )

                if observed != expected:
                    mismatch.append(col)

        row["n_pillar_onehot_mismatches"] = len(mismatch)
        row["mismatch_columns"] = "|".join(mismatch)
        row["pillar_onehot_match"] = len(mismatch) == 0

        pillar_parity_rows.append(row)

    pillar_parity_df = pd.DataFrame(
        pillar_parity_rows
    )

    pillar_parity_df.to_csv(
        OUT / "V4_V2_DYNAMIC_PILLAR_PARITY.csv",
        index=False,
    )

    print("Optional V2 dynamic pillar parity:")
    display(
        pillar_parity_df.agg({
            "pillar_onehot_match": ["count", "sum"],
            "n_pillar_onehot_mismatches": "sum",
        })
    )

else:
    pillar_parity_df = pd.DataFrame()
    print(
        "Optional all-feature V2 cache not found; "
        "skipping dynamic pillar one-hot audit."
    )

Optional V2 dynamic pillar parity:


,pillar_onehot_match,n_pillar_onehot_mismatches
count,216,NaN
sum,216,0.0


## 6. Mismatch localization

If a row fails, show whether the historical cache implies a different DW/SW group than the current engine.

In [8]:
def historical_group(row, prefix):
    active = []

    for group in GROUPS:
        col = "historical__tengod__%s_group_%s" % (
            prefix,
            group,
        )

        if col in row.index and float(row[col]) == 1.0:
            active.append(group)

    return active[0] if len(active) == 1 else "|".join(active)

if len(strict_parity):
    strict_parity["historical_dw_group"] = strict_parity.apply(
        lambda r: historical_group(r, "dw"),
        axis=1,
    )

    strict_parity["historical_sw_group"] = strict_parity.apply(
        lambda r: historical_group(r, "sw"),
        axis=1,
    )

    mismatch_detail = strict_parity[
        ~strict_parity[
            "strict_tengod_row_match"
        ]
    ][
        [
            "pool",
            "subject_id",
            "name",
            "year",
            "canonical_day_stem",
            "canonical_dw_stem",
            "canonical_dw_branch",
            "canonical_sw_stem",
            "canonical_sw_branch",
            "historical_dw_group",
            "canonical_dw_group",
            "historical_sw_group",
            "canonical_sw_group",
            "n_tengod_mismatches",
            "mismatch_features",
        ]
    ].copy()

    mismatch_detail.to_csv(
        OUT / "V4_TENGOD_PARITY_MISMATCH_DETAIL.csv",
        index=False,
    )

    print("Strict mismatched rows:", len(mismatch_detail))
    if len(mismatch_detail):
        display(mismatch_detail.head(100))

Strict mismatched rows: 0


## 7. Strict parity summary and decision

Decision policy:

### `PASS_MODERN_STRICT_PARITY_V1_LEGACY_UNVERIFIED`
- V2 NEW_DEV: 0 frozen-feature mismatches
- NEW_DEV_2: 0 frozen-feature mismatches
- V1 legacy cache invariants pass
- V1 canonical parity remains explicitly unclaimed

### `FAIL_MODERN_FEATURE_PARITY`
- at least one strict mismatch in V2 NEW_DEV or NEW_DEV_2

### `FAIL_V1_LEGACY_CACHE_INVARIANTS`
- modern strict parity passes, but historical V1 cache itself violates basic feature invariants

## Consequence

If modern strict parity fails:
**stop all V4 modeling and resolve the engine/cache lineage first.**

If modern strict parity passes but V1 remains unverifiable:
either create a verified V1 engine-input reconstruction package or exclude V1 legacy rows from a canonical V4 model.

In [9]:
summary_rows = []

for pool, g in strict_parity.groupby("pool"):
    summary_rows.append({
        "pool": pool,
        "rows": len(g),
        "subjects": g["subject_id"].nunique(),
        "exact_match_rows": int(
            g["strict_tengod_row_match"].sum()
        ),
        "mismatched_rows": int(
            (~g["strict_tengod_row_match"]).sum()
        ),
        "total_feature_mismatches": int(
            g["n_tengod_mismatches"].sum()
        ),
        "exact_row_match_rate": float(
            g["strict_tengod_row_match"].mean()
        ),
    })

parity_summary = pd.DataFrame(
    summary_rows
)

parity_summary.to_csv(
    OUT / "V4_STRICT_TENGOD_PARITY_SUMMARY.csv",
    index=False,
)

display(parity_summary)

modern_mismatches = int(
    strict_parity[
        "n_tengod_mismatches"
    ].sum()
)

if modern_mismatches > 0:
    status = "FAIL_MODERN_FEATURE_PARITY"
elif not V1_LEGACY_INVARIANTS_PASS:
    status = "FAIL_V1_LEGACY_CACHE_INVARIANTS"
else:
    status = "PASS_MODERN_STRICT_PARITY_V1_LEGACY_UNVERIFIED"

decision = {
    "version": "SAJU_ML_FEATURE_PARITY_AUDIT_DECISION_V4",
    "notebook_version": NOTEBOOK_VERSION,
    "created_at": datetime.now().isoformat(
        timespec="seconds"
    ),
    "status": status,
    "engine": {
        "saju_engine_sha256": sha256_file(
            ENGINE_PATH
        ),
        "sajupy_version": getattr(
            sajupy,
            "__version__",
            "unknown",
        ),
        "engine_year_range": [
            ENGINE_MIN_YEAR,
            ENGINE_MAX_YEAR,
        ],
    },
    "strict_modern_parity": {
        row["pool"]: {
            "rows": int(row["rows"]),
            "subjects": int(row["subjects"]),
            "mismatched_rows": int(
                row["mismatched_rows"]
            ),
            "total_feature_mismatches": int(
                row["total_feature_mismatches"]
            ),
            "exact_row_match_rate": float(
                row["exact_row_match_rate"]
            ),
        }
        for row in summary_rows
    },
    "v1_legacy": {
        "cache_sha256": sha256_file(
            v1_cache_path
        ),
        "invariants_pass": (
            V1_LEGACY_INVARIANTS_PASS
        ),
        "canonical_parity_claimed": False,
        "status": (
            "LEGACY_CANONICAL_PARITY_NOT_CLAIMED"
        ),
    },
    "holdout_integrity": {
        "NEW_CONFIRM_loaded": False,
        "Public_CHECK_loaded": False,
        "Public_FINAL_loaded": False,
        "Validation_B_loaded": False,
    },
    "next_rule": (
        "If modern parity fails, stop and repair feature lineage before any V4 modeling. "
        "If modern parity passes, combine this result with the V4 data-integrity audit. "
        "Do not start a V4 feature tournament until critical/high data defects are resolved."
    ),
}

with open(
    OUT / "V4_FEATURE_PARITY_AUDIT_DECISION.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        decision,
        f,
        ensure_ascii=False,
        indent=2,
    )

print(json.dumps(decision, ensure_ascii=False, indent=2))

,pool,rows,subjects,exact_match_rows,mismatched_rows,total_feature_mismatches,exact_row_match_rate
0,NEW_DEV_2_CONSUMED,200,100,200,0,0,1.0
1,V2_NEW_DEV,216,70,216,0,0,1.0


{
  "version": "SAJU_ML_FEATURE_PARITY_AUDIT_DECISION_V4",
  "notebook_version": "SAJU_ML_FEATURE_PARITY_AUDIT_V4_20260815",
  "created_at": "2026-08-15T22:23:32",
  "status": "PASS_MODERN_STRICT_PARITY_V1_LEGACY_UNVERIFIED",
  "engine": {
    "saju_engine_sha256": "d39e0c4d777ae3a19394c9175319a4f8a6a709c2b4d34400965b618ffbfdad1e",
    "sajupy_version": "0.2.0",
    "engine_year_range": [
      1900,
      2100
    ]
  },
  "strict_modern_parity": {
    "NEW_DEV_2_CONSUMED": {
      "rows": 200,
      "subjects": 100,
      "mismatched_rows": 0,
      "total_feature_mismatches": 0,
      "exact_row_match_rate": 1.0
    },
    "V2_NEW_DEV": {
      "rows": 216,
      "subjects": 70,
      "mismatched_rows": 0,
      "total_feature_mismatches": 0,
      "exact_row_match_rate": 1.0
    }
  },
  "v1_legacy": {
    "cache_sha256": "d57fdb7d390ae5b11500a06fa479ecbd25bcb5899d9d1d43caaa5cd5865ad09c",
    "invariants_pass": true,
    "canonical_parity_claimed": false,
    "status": "LEGACY_CANO

## Files to send back

After **Restart Kernel -> Run All**, send:

```text
research/ml/artifacts/v4_feature_parity_audit/
├── V4_FEATURE_PARITY_AUDIT_DECISION.json
├── V4_STRICT_TENGOD_PARITY_SUMMARY.csv
├── V4_TENGOD_PARITY_MISMATCH_DETAIL.csv
├── V4_V1_LEGACY_CACHE_INVARIANTS.csv
└── V4_V2_DYNAMIC_PILLAR_PARITY.csv   # if optional V2 all-feature cache exists
```

The most important output is the decision JSON.